# Access Control & Audit Trails Demo

Demonstrates document-level access control and audit logging patterns for RAG systems.

**Key Concepts:**
- Pre-filter ACLs at retrieval time
- Metadata-based permission enforcement
- Audit trail logging with Haystack tracing

**Prerequisites:**
```bash
pip install qdrant-client haystack-ai sentence-transformers
```

In [17]:
from qdrant_client import QdrantClient, models
from sentence_transformers import SentenceTransformer
import uuid

# Initialize
client = QdrantClient(":memory:")
embedder = SentenceTransformer("all-MiniLM-L6-v2")

print("✓ Qdrant and embedder initialized")

✓ Qdrant and embedder initialized


---

## 1. Documents with Access Control Metadata

Each document includes `allowed_roles` and `allowed_users` for permission filtering.

In [18]:
# Sample documents with ACL metadata
DOCUMENTS = [
    {
        "content": "Q3 revenue projections show 15% growth year-over-year.",
        "allowed_roles": ["finance", "executive"],
        "allowed_users": [],
        "classification": "confidential"
    },
    {
        "content": "Employee health benefits include dental and vision coverage.",
        "allowed_roles": ["hr", "employee"],
        "allowed_users": [],
        "classification": "internal"
    },
    {
        "content": "Company holiday schedule: Dec 24-Jan 2 office closed.",
        "allowed_roles": ["employee"],  # All employees
        "allowed_users": [],
        "classification": "public"
    },
    {
        "content": "Board meeting notes: Acquisition target identified - Project Phoenix.",
        "allowed_roles": ["executive"],
        "allowed_users": ["user_ceo", "user_cfo"],
        "classification": "top-secret"
    },
    {
        "content": "Engineering roadmap: New AI features planned for Q2.",
        "allowed_roles": ["engineering", "product"],
        "allowed_users": [],
        "classification": "internal"
    }
]

print(f"Prepared {len(DOCUMENTS)} documents with ACL metadata")
for doc in DOCUMENTS:
    print(f"  - {doc['classification']}: roles={doc['allowed_roles']}")

Prepared 5 documents with ACL metadata
  - confidential: roles=['finance', 'executive']
  - internal: roles=['hr', 'employee']
  - public: roles=['employee']
  - top-secret: roles=['executive']
  - internal: roles=['engineering', 'product']


In [19]:
# Create collection and index documents
COLLECTION = "acl_documents"

client.create_collection(
    collection_name=COLLECTION,
    vectors_config=models.VectorParams(
        size=384,  # all-MiniLM-L6-v2 dimension
        distance=models.Distance.COSINE
    )
)

# Embed and upsert
points = []
for i, doc in enumerate(DOCUMENTS):
    embedding = embedder.encode(doc["content"]).tolist()
    points.append(models.PointStruct(
        id=i,
        vector=embedding,
        payload=doc
    ))

client.upsert(collection_name=COLLECTION, points=points)
print(f"✓ Indexed {len(points)} documents with ACL metadata")

✓ Indexed 5 documents with ACL metadata


---

## 2. Pre-Filter ACL Pattern

Filter documents at query time based on user permissions. Unauthorized documents never reach the LLM.

In [20]:
def search_with_acl(
    query: str,
    user_roles: list[str],
    user_id: str,
    limit: int = 3
) -> list[dict]:
    """
    Search with access control filtering.
    
    Documents are returned only if:
    - User has a matching role in allowed_roles, OR
    - User ID is in allowed_users
    """
    query_embedding = embedder.encode(query).tolist()
    
    # Build ACL filter: role match OR user match
    acl_filter = models.Filter(
        should=[  # OR condition
            models.FieldCondition(
                key="allowed_roles",
                match=models.MatchAny(any=user_roles)
            ),
            models.FieldCondition(
                key="allowed_users",
                match=models.MatchValue(value=user_id)
            ),
        ]
    )
    
    results = client.query_points(
        collection_name=COLLECTION,
        query=query_embedding,
        query_filter=acl_filter,
        limit=limit
    )
    
    return [
        {
            "content": r.payload["content"],
            "classification": r.payload["classification"],
            "score": r.score
        }
        for r in results.points
    ]

print("✓ ACL search function defined")

✓ ACL search function defined


In [21]:
# Test: Regular employee searching for company info
print("User: Regular Employee (roles=['employee'])")
print("Query: 'What are the company benefits?'")
print("=" * 60)

results = search_with_acl(
    query="What are the company benefits?",
    user_roles=["employee"],
    user_id="user_123"
)

for r in results:
    print(f"[{r['classification']}] {r['content'][:60]}...")

print(f"\n→ Employee sees {len(results)} documents (no finance/executive docs)")

User: Regular Employee (roles=['employee'])
Query: 'What are the company benefits?'
[internal] Employee health benefits include dental and vision coverage....
[public] Company holiday schedule: Dec 24-Jan 2 office closed....

→ Employee sees 2 documents (no finance/executive docs)


In [22]:
# Test: Finance manager searching for revenue
print("User: Finance Manager (roles=['employee', 'finance'])")
print("Query: 'What are the revenue projections?'")
print("=" * 60)

results = search_with_acl(
    query="What are the revenue projections?",
    user_roles=["employee", "finance"],
    user_id="user_456"
)

for r in results:
    print(f"[{r['classification']}] {r['content'][:60]}...")

print(f"\n→ Finance manager sees {len(results)} documents (including confidential)")

User: Finance Manager (roles=['employee', 'finance'])
Query: 'What are the revenue projections?'
[confidential] Q3 revenue projections show 15% growth year-over-year....
[public] Company holiday schedule: Dec 24-Jan 2 office closed....
[internal] Employee health benefits include dental and vision coverage....

→ Finance manager sees 3 documents (including confidential)


In [23]:
# Test: CEO with explicit user access to top-secret
print("User: CEO (roles=['executive'], user_id='user_ceo')")
print("Query: 'What is Project Phoenix?'")
print("=" * 60)

results = search_with_acl(
    query="What is Project Phoenix?",
    user_roles=["executive"],
    user_id="user_ceo"
)

for r in results:
    print(f"[{r['classification']}] {r['content'][:60]}...")

print(f"\n→ CEO sees {len(results)} documents (including top-secret via user_id match)")

User: CEO (roles=['executive'], user_id='user_ceo')
Query: 'What is Project Phoenix?'
[top-secret] Board meeting notes: Acquisition target identified - Project...
[confidential] Q3 revenue projections show 15% growth year-over-year....

→ CEO sees 2 documents (including top-secret via user_id match)


In [24]:
# Test: Regular executive WITHOUT explicit user access
print("User: VP (roles=['executive'], user_id='user_vp')")
print("Query: 'What is Project Phoenix?'")
print("=" * 60)

results = search_with_acl(
    query="What is Project Phoenix?",
    user_roles=["executive"],
    user_id="user_vp"
)

for r in results:
    print(f"[{r['classification']}] {r['content'][:60]}...")

# Note: VP sees top-secret because 'executive' role is in allowed_roles
print(f"\n→ VP sees top-secret doc (executive role in allowed_roles)")

User: VP (roles=['executive'], user_id='user_vp')
Query: 'What is Project Phoenix?'
[top-secret] Board meeting notes: Acquisition target identified - Project...
[confidential] Q3 revenue projections show 15% growth year-over-year....

→ VP sees top-secret doc (executive role in allowed_roles)


---

## 3. Audit Trail Logging

Log all retrieval operations for compliance and security auditing.

In [25]:
import logging
import json
from datetime import datetime

# Configure audit logger (in production: send to immutable log store)
audit_logger = logging.getLogger("rag.audit")
audit_logger.setLevel(logging.INFO)
handler = logging.StreamHandler()
handler.setFormatter(logging.Formatter('%(message)s'))
audit_logger.addHandler(handler)

def search_with_audit(
    query: str,
    user_roles: list[str],
    user_id: str,
    limit: int = 3
) -> list[dict]:
    """Search with ACL filtering and audit logging."""
    start_time = datetime.now()
    
    # Perform search
    results = search_with_acl(query, user_roles, user_id, limit)
    
    elapsed_ms = (datetime.now() - start_time).total_seconds() * 1000
    
    # Log audit record (append-only, immutable in production)
    audit_record = {
        "timestamp": start_time.isoformat(),
        "user_id": user_id,
        "user_roles": user_roles,
        "query_hash": hash(query) % 10**8,  # Don't log raw query for privacy
        "retrieved_count": len(results),
        "retrieved_classifications": [r["classification"] for r in results],
        "latency_ms": round(elapsed_ms, 2)
    }
    
    audit_logger.info(f"AUDIT: {json.dumps(audit_record)}")
    
    return results

print("✓ Audit-enabled search function defined")

✓ Audit-enabled search function defined


In [26]:
# Test audit logging
print("Audit Trail Demo")
print("=" * 60)

# Simulate multiple users
users = [
    {"id": "user_123", "roles": ["employee"], "query": "holiday schedule"},
    {"id": "user_456", "roles": ["finance"], "query": "revenue projections"},
    {"id": "user_ceo", "roles": ["executive"], "query": "acquisition plans"},
]

for user in users:
    results = search_with_audit(
        query=user["query"],
        user_roles=user["roles"],
        user_id=user["id"]
    )
    print()

Audit Trail Demo


AUDIT: {"timestamp": "2026-01-27T03:33:26.060004", "user_id": "user_123", "user_roles": ["employee"], "query_hash": 48355416, "retrieved_count": 2, "retrieved_classifications": ["public", "internal"], "latency_ms": 8.9}
AUDIT: {"timestamp": "2026-01-27T03:33:26.060004", "user_id": "user_123", "user_roles": ["employee"], "query_hash": 48355416, "retrieved_count": 2, "retrieved_classifications": ["public", "internal"], "latency_ms": 8.9}
AUDIT: {"timestamp": "2026-01-27T03:33:26.069377", "user_id": "user_456", "user_roles": ["finance"], "query_hash": 18490476, "retrieved_count": 1, "retrieved_classifications": ["confidential"], "latency_ms": 8.44}
AUDIT: {"timestamp": "2026-01-27T03:33:26.069377", "user_id": "user_456", "user_roles": ["finance"], "query_hash": 18490476, "retrieved_count": 1, "retrieved_classifications": ["confidential"], "latency_ms": 8.44}
AUDIT: {"timestamp": "2026-01-27T03:33:26.078368", "user_id": "user_ceo", "user_roles": ["executive"], "query_hash": 43028895, "retr

---

## 4. Haystack Integration with Tracing

Using Haystack's built-in tracing for observability.

In [27]:
from haystack import Pipeline, Document, component
from haystack.components.retrievers.in_memory import InMemoryEmbeddingRetriever
from haystack.document_stores.in_memory import InMemoryDocumentStore
from haystack.components.embedders import SentenceTransformersTextEmbedder, SentenceTransformersDocumentEmbedder
from haystack import tracing

# Enable content tracing for audit purposes
tracing.tracer.is_content_tracing_enabled = True

print("✓ Haystack tracing enabled")

✓ Haystack tracing enabled


In [28]:
# Create document store with ACL metadata
doc_store = InMemoryDocumentStore()

haystack_docs = [
    Document(
        content=doc["content"],
        meta={
            "allowed_roles": doc["allowed_roles"],
            "allowed_users": doc["allowed_users"],
            "classification": doc["classification"]
        }
    )
    for doc in DOCUMENTS
]

# Embed documents
doc_embedder = SentenceTransformersDocumentEmbedder(model="all-MiniLM-L6-v2")
doc_embedder.warm_up()
embedded_docs = doc_embedder.run(haystack_docs)["documents"]
doc_store.write_documents(embedded_docs)

print(f"✓ Indexed {len(embedded_docs)} documents in Haystack store")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

✓ Indexed 5 documents in Haystack store


In [29]:
@component
class ACLFilterComponent:
    """
    Haystack component that filters documents based on user permissions.
    Applied post-retrieval but BEFORE LLM generation.
    """
    
    @component.output_types(documents=list[Document])
    def run(self, documents: list[Document], user_roles: list[str], user_id: str):
        authorized = []
        
        for doc in documents:
            allowed_roles = doc.meta.get("allowed_roles", [])
            allowed_users = doc.meta.get("allowed_users", [])
            
            # Check role match or user match
            role_match = any(role in allowed_roles for role in user_roles)
            user_match = user_id in allowed_users
            
            if role_match or user_match:
                authorized.append(doc)
        
        return {"documents": authorized}

print("✓ ACL filter component defined")

✓ ACL filter component defined


---

## 5. Secure Retriever Wrapper Pattern

A cleaner approach: wrap the retriever to inject ACL filters automatically. The caller just passes user context, and filtering happens transparently.

In [30]:
@component
class SecureRetriever:
    """
    Wrapper that combines embedding + retrieval + ACL filtering in one component.
    
    Benefits:
    - Single component handles security
    - Caller just provides query text and user context
    - ACL logic is encapsulated and reusable
    """
    
    def __init__(self, document_store, model: str = "all-MiniLM-L6-v2", top_k: int = 5):
        self.document_store = document_store
        self.embedder = SentenceTransformersTextEmbedder(model=model)
        self.top_k = top_k
        
    def warm_up(self):
        self.embedder.warm_up()
    
    @component.output_types(documents=list[Document], query=str)
    def run(self, query: str, user_roles: list[str], user_id: str):
        # Step 1: Embed the query
        embedding_result = self.embedder.run(text=query)
        query_embedding = embedding_result["embedding"]
        
        # Step 2: Retrieve all candidates
        all_docs = self.document_store.embedding_retrieval(
            query_embedding=query_embedding,
            top_k=self.top_k * 2  # Retrieve extra to account for filtering
        )
        
        # Step 3: Apply ACL filter
        authorized = []
        for doc in all_docs:
            allowed_roles = doc.meta.get("allowed_roles", [])
            allowed_users = doc.meta.get("allowed_users", [])
            
            role_match = any(role in allowed_roles for role in user_roles)
            user_match = user_id in allowed_users
            
            if role_match or user_match:
                authorized.append(doc)
                if len(authorized) >= self.top_k:
                    break
        
        return {"documents": authorized, "query": query}

print("✓ SecureRetriever component defined")

✓ SecureRetriever component defined


In [31]:
# Build a complete RAG pipeline with SecureRetriever
from haystack.components.builders import PromptBuilder

# Define prompt template
PROMPT_TEMPLATE = """
Answer the question based on the provided context.
If the context doesn't contain relevant information, say so.

Context:
{% for doc in documents %}
- {{ doc.content }}
{% endfor %}

Question: {{ query }}

Answer:
"""

# Create pipeline
secure_pipeline = Pipeline()

# Add components
secure_retriever = SecureRetriever(document_store=doc_store, top_k=3)
secure_retriever.warm_up()

secure_pipeline.add_component("secure_retriever", secure_retriever)
secure_pipeline.add_component("prompt_builder", PromptBuilder(
    template=PROMPT_TEMPLATE,
    required_variables=["documents", "query"]
))

# Connect components
secure_pipeline.connect("secure_retriever.documents", "prompt_builder.documents")
secure_pipeline.connect("secure_retriever.query", "prompt_builder.query")

print("✓ Secure RAG pipeline ready (without LLM for demo)")
print("\nPipeline structure:")
print("  query + user_context → SecureRetriever → PromptBuilder → (LLM)")
print("                         ↑")
print("                   ACL filtering happens here")

✓ Secure RAG pipeline ready (without LLM for demo)

Pipeline structure:
  query + user_context → SecureRetriever → PromptBuilder → (LLM)
                         ↑
                   ACL filtering happens here


In [32]:
# Test 1: Regular employee asking about financials
print("Test 1: Employee asking about revenue")
print("=" * 60)

result = secure_pipeline.run(
    {
        "secure_retriever": {
            "query": "What are the revenue projections?",
            "user_roles": ["employee"],
            "user_id": "user_123"
        }
    },
    include_outputs_from={"secure_retriever", "prompt_builder"}
)

print(f"User: employee (user_123)")
print(f"Query: 'What are the revenue projections?'")
print(f"\nDocuments retrieved: {len(result['secure_retriever']['documents'])}")

for doc in result["secure_retriever"]["documents"]:
    print(f"  [{doc.meta['classification']}] {doc.content[:50]}...")

print(f"\n→ Employee cannot see confidential financial data")
print(f"\nPrompt that would go to LLM:")
print("-" * 40)
print(result["prompt_builder"]["prompt"][:500])

Test 1: Employee asking about revenue


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

User: employee (user_123)
Query: 'What are the revenue projections?'

Documents retrieved: 2
  [public] Company holiday schedule: Dec 24-Jan 2 office clos...
  [internal] Employee health benefits include dental and vision...

→ Employee cannot see confidential financial data

Prompt that would go to LLM:
----------------------------------------

Answer the question based on the provided context.
If the context doesn't contain relevant information, say so.

Context:

- Company holiday schedule: Dec 24-Jan 2 office closed.

- Employee health benefits include dental and vision coverage.


Question: What are the revenue projections?

Answer:


In [33]:
# Test 2: Finance manager asking about revenue
print("Test 2: Finance manager asking about revenue")
print("=" * 60)

result = secure_pipeline.run(
    {
        "secure_retriever": {
            "query": "What are the revenue projections?",
            "user_roles": ["employee", "finance"],
            "user_id": "user_456"
        }
    },
    include_outputs_from={"secure_retriever", "prompt_builder"}
)

print(f"User: finance manager (user_456)")
print(f"Query: 'What are the revenue projections?'")
print(f"\nDocuments retrieved: {len(result['secure_retriever']['documents'])}")

for doc in result["secure_retriever"]["documents"]:
    print(f"  [{doc.meta['classification']}] {doc.content[:50]}...")

print(f"\n→ Finance manager CAN see confidential financial data")
print(f"\nPrompt that would go to LLM:")
print("-" * 40)
print(result["prompt_builder"]["prompt"][:500])

Test 2: Finance manager asking about revenue


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

User: finance manager (user_456)
Query: 'What are the revenue projections?'

Documents retrieved: 3
  [confidential] Q3 revenue projections show 15% growth year-over-y...
  [public] Company holiday schedule: Dec 24-Jan 2 office clos...
  [internal] Employee health benefits include dental and vision...

→ Finance manager CAN see confidential financial data

Prompt that would go to LLM:
----------------------------------------

Answer the question based on the provided context.
If the context doesn't contain relevant information, say so.

Context:

- Q3 revenue projections show 15% growth year-over-year.

- Company holiday schedule: Dec 24-Jan 2 office closed.

- Employee health benefits include dental and vision coverage.


Question: What are the revenue projections?

Answer:


In [34]:
# Test 3: CEO asking about acquisition (top-secret)
print("Test 3: CEO asking about Project Phoenix")
print("=" * 60)

result = secure_pipeline.run(
    {
        "secure_retriever": {
            "query": "Tell me about Project Phoenix acquisition",
            "user_roles": ["executive"],
            "user_id": "user_ceo"
        }
    },
    include_outputs_from={"secure_retriever", "prompt_builder"}
)

print(f"User: CEO (user_ceo)")
print(f"Query: 'Tell me about Project Phoenix acquisition'")
print(f"\nDocuments retrieved: {len(result['secure_retriever']['documents'])}")

for doc in result["secure_retriever"]["documents"]:
    print(f"  [{doc.meta['classification']}] {doc.content[:50]}...")

print(f"\n→ CEO CAN see top-secret data (via executive role + user_id)")
print(f"\nPrompt that would go to LLM:")
print("-" * 40)
print(result["prompt_builder"]["prompt"][:500])

Test 3: CEO asking about Project Phoenix


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

User: CEO (user_ceo)
Query: 'Tell me about Project Phoenix acquisition'

Documents retrieved: 2
  [top-secret] Board meeting notes: Acquisition target identified...
  [confidential] Q3 revenue projections show 15% growth year-over-y...

→ CEO CAN see top-secret data (via executive role + user_id)

Prompt that would go to LLM:
----------------------------------------

Answer the question based on the provided context.
If the context doesn't contain relevant information, say so.

Context:

- Board meeting notes: Acquisition target identified - Project Phoenix.

- Q3 revenue projections show 15% growth year-over-year.


Question: Tell me about Project Phoenix acquisition

Answer:


In [35]:
# Test 4: Comparison - same query, different access levels
print("Test 4: Access Comparison - Same Query, Different Users")
print("=" * 60)

test_cases = [
    {"name": "Intern", "roles": ["employee"], "user_id": "user_intern"},
    {"name": "Engineer", "roles": ["employee", "engineering"], "user_id": "user_eng"},
    {"name": "CFO", "roles": ["executive", "finance"], "user_id": "user_cfo"},
]

query = "What are all the company plans?"

for user in test_cases:
    result = secure_pipeline.run(
        {
            "secure_retriever": {
                "query": query,
                "user_roles": user["roles"],
                "user_id": user["user_id"]
            }
        },
        include_outputs_from={"secure_retriever"}
    )
    
    docs = result["secure_retriever"]["documents"]
    classifications = [d.meta["classification"] for d in docs]
    
    print(f"\n{user['name']} ({user['roles']}):")
    print(f"  Can see: {classifications}")

Test 4: Access Comparison - Same Query, Different Users


Batches:   0%|          | 0/1 [00:00<?, ?it/s]


Intern (['employee']):
  Can see: ['internal', 'public']


Batches:   0%|          | 0/1 [00:00<?, ?it/s]


Engineer (['employee', 'engineering']):
  Can see: ['internal', 'internal', 'public']


Batches:   0%|          | 0/1 [00:00<?, ?it/s]


CFO (['executive', 'finance']):
  Can see: ['top-secret', 'confidential']


### SecureRetriever vs ACLFilterComponent

| Aspect | SecureRetriever (Section 5) | ACLFilterComponent (Section 4) |
|--------|----------------------------|-------------------------------|
| **Encapsulation** | All-in-one: embed + retrieve + filter | Separate components in pipeline |
| **Reusability** | Single component to add to any pipeline | Need to wire 3 components |
| **Flexibility** | Less flexible, opinionated | More flexible, composable |
| **Best for** | Standard ACL patterns | Complex/custom permission logic |

**Recommendation**: Use `SecureRetriever` for most cases. Use separate `ACLFilterComponent` when you need to share the retriever across pipelines with different filtering logic.

In [36]:
# Build pipeline with ACL filtering
pipeline = Pipeline()

pipeline.add_component("embedder", SentenceTransformersTextEmbedder(model="all-MiniLM-L6-v2"))
pipeline.add_component("retriever", InMemoryEmbeddingRetriever(document_store=doc_store, top_k=5))
pipeline.add_component("acl_filter", ACLFilterComponent())

pipeline.connect("embedder.embedding", "retriever.query_embedding")
pipeline.connect("retriever.documents", "acl_filter.documents")

print("✓ Haystack pipeline with ACL filter ready")

✓ Haystack pipeline with ACL filter ready


In [37]:
# Test: Employee query
print("Haystack Pipeline - Employee Query")
print("=" * 60)

result = pipeline.run({
    "embedder": {"text": "company benefits and schedule"},
    "acl_filter": {
        "user_roles": ["employee"],
        "user_id": "user_123"
    }
})

print(f"Retrieved (before filter): 5 documents")
print(f"Authorized (after filter): {len(result['acl_filter']['documents'])} documents")
print()
for doc in result['acl_filter']['documents']:
    print(f"  [{doc.meta['classification']}] {doc.content[:50]}...")

Haystack Pipeline - Employee Query


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Retrieved (before filter): 5 documents
Authorized (after filter): 2 documents

  [internal] Employee health benefits include dental and vision...
  [public] Company holiday schedule: Dec 24-Jan 2 office clos...


In [38]:
# Test: Executive query
print("Haystack Pipeline - Executive Query")
print("=" * 60)

result = pipeline.run({
    "embedder": {"text": "acquisition and revenue"},
    "acl_filter": {
        "user_roles": ["executive"],
        "user_id": "user_ceo"
    }
})

print(f"Retrieved (before filter): 5 documents")
print(f"Authorized (after filter): {len(result['acl_filter']['documents'])} documents")
print()
for doc in result['acl_filter']['documents']:
    print(f"  [{doc.meta['classification']}] {doc.content[:50]}...")

Haystack Pipeline - Executive Query


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Retrieved (before filter): 5 documents
Authorized (after filter): 2 documents

  [top-secret] Board meeting notes: Acquisition target identified...
  [confidential] Q3 revenue projections show 15% growth year-over-y...


---

## Summary

**Access Control Patterns:**

| Pattern | When to Use | Trade-offs |
|---------|-------------|------------|
| Pre-filter ACLs | Simple role/user checks | Requires metadata at ingestion |
| Post-retrieval filter | Complex permission logic | Retrieves then filters (less efficient) |
| Hybrid | Best of both | More implementation complexity |

**Key Principles:**
1. Enforce permissions BEFORE documents reach the LLM
2. Store ACL metadata with document embeddings
3. Log all access for audit compliance
4. Never rely on LLM to filter sensitive content

**Frameworks to Consider:**
- **Permit.io**: Fine-grained authorization with LangChain integration
- **Cerbos**: Enterprise authorization with compliance features
- **Langfuse/LangSmith**: Observability and audit trails
- **OpenTelemetry**: Standard protocol for distributed tracing